# Loading satellite data directly in Python

**Find real satellite scenes by place, time and cloudiness, and read just the part you need - straight over HTTP, with no full-scene downloads.**

<a href="https://colab.research.google.com/github/tommyscodebase/easy-eo-tutorials/blob/main/02-fetching-satellite-data/01_stac_search_and_load.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>
<a href="https://www.youtube.com/playlist?list=PLQDjJNQh9NXU"><img src="https://img.shields.io/badge/YouTube-Watch%20the%20series-FF0000?logo=youtube&logoColor=white" alt="Watch the series on YouTube"></a>
<a href="https://github.com/Tommy-Burns/easy-eo"><img src="https://img.shields.io/badge/GitHub-easy--eo-181717?logo=github&logoColor=white" alt="easy-eo on GitHub"></a>
<a href="https://easy-eo.readthedocs.io"><img src="https://img.shields.io/badge/docs-easy--eo.readthedocs.io-8CA1AF?logo=readthedocs&logoColor=white" alt="Documentation"></a>
<a href="https://pypi.org/project/easy-eo/"><img src="https://img.shields.io/pypi/v/easy-eo.svg" alt="PyPI"></a>

---

This episode is part of the [Easy-EO Tutorial series](https://www.youtube.com/playlist?list=YOUR_PLAYLIST_ID).

In the next few minutes you will:

1. **Search a live STAC catalog** by place, time and cloud cover - metadata only, no pixels read
2. **Inspect a scene** and load just the bands you need, straight over HTTP
3. **Go straight into analysis**, computing and plotting NDVI from the loaded scene

- **Data:** Sentinel-2 L2A from Microsoft Planetary Computer
- **Network: required.** This notebook talks to a live catalog, so a network connection is required if you are running it locally.

### Before you run this
easy-eo must be installed, with the STAC extra: `pip install "easy-eo[stac]"`. <a href="https://youtu.be/7neK_fxiFFM"><img src="https://img.shields.io/badge/Watch%20this%20video-FF0000?logo=youtube&logoColor=white" alt="Watch this video"></a> or check [the installation instructions](https://github.com/tommyscodebase/easy-eo-tutorials#setup) here


📓 All notebooks in the series: [tommyscodebase/easy-eo-tutorials](https://github.com/tommyscodebase/easy-eo-tutorials) · 🐛 Found a bug in the library? [Open an issue](https://github.com/Tommy-Burns/easy-eo/issues)

In [ ]:
# Running in Colab? Install Easy-EO. Does nothing anywhere else.
import sys

if "google.colab" in sys.modules:
    %pip install -q "easy-eo[stac]"

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# Keeps the figures stored in this notebook small. raise it for print-quality output.
plt.rcParams["figure.dpi"] = 80

In [ ]:
import eeo

eeo.show_versions()

## Searching

`stac_search` queries a STAC catalog and returns matching scenes. It fetches **metadata only** - no pixels are read and nothing is downloaded.

The bounding box is always in **WGS 84 lon/lat degrees** (`minx, miny, maxx, maxy`), whatever CRS the imagery itself uses.
You can create a bounding box for a different place using this tool [Mofei Dev Tools Bbox](https://tools.mofei.life/bbox#1/0/0)

In [ ]:
# An area in Passau, Germany
bbox = (13.37, 48.22, 13.47, 48.28)

results = eeo.stac_search(
    "sentinel-2-l2a",
    # catalog="https://earth-search.aws.element84.com/v1",
    bbox=bbox,
    datetime="2026-08-01/..",
    cloud_cover=10,
    limit=10,
)

print(f"{len(results)} scenes")
print("catalog:", results.catalog)

Results come back **oldest first**, each carrying its acquisition timestamp - the ordering a time series needs.

- `cloud_cover` - maximum percent cloud, applied by the catalog against each scene's own `eo:cloud_cover` value.
- `limit` - maximum number of items returned.
- `intersects` - a shape instead of a rectangle: a GeoDataFrame, a shapely geometry, GeoJSON, or a path to any vector file. Cannot be combined with `bbox`.

In [ ]:
for item in results:
    print(f"{results.index(item)} {item.timestamp:%Y-%m-%d}  cloud {item.cloud_cover:5.2f}%  {item.id}")

Two catalog behaviours are worth knowing rather than being surprised by: the cloud filter is applied to the catalog's own indexed value, so a scene can come back slightly above your threshold; and catalogs return **reprocessed duplicates** of  the same acquisition, which is why you may see one date twice.

In [ ]:
from collections import Counter

dates = Counter(item.timestamp.date() for item in results)
for date, count in sorted(dates.items()):
    marker = "  <- duplicate acquisition" if count > 1 else ""
    print(f"{date}  x{count}{marker}")

### Searching with a shape

### Loading a sample vector file file the sample datasets

In [ ]:
import geopandas as gpd
from eeo.datasets import load_sample_dataset

sd = load_sample_dataset()
aoi = gpd.read_file(str(sd.boundary.path))
print(type(aoi))

aoi.plot()

In [ ]:
shaped = eeo.stac_search(
    "sentinel-2-l2a",
    catalog="https://earth-search.aws.element84.com/v1",
    intersects=aoi,
    datetime="2026-08-01/..",
    cloud_cover=20,
    limit=5,
)
print(f"{len(shaped)} scenes intersecting the sample ROI")

## Inspecting an item

Each result wraps the underlying `pystac.Item` (still reachable at `.item`) with the properties you actually want.

In [ ]:
item = shaped[-1]

print("id         :", item.id)
print("collection :", item.collection)
print("timestamp  :", item.timestamp)
print("cloud      :", item.cloud_cover)
print("bbox       :", item.bbox)
print("band names :", item.asset_names)

## Loading pixels

`load()` reads the assets you name into a single `EEORasterDataset`.

In [ ]:
# scene = item.load(["B02", "B03", "B04", "B08"])
scene = item.load(["blue", "green", "red", "nir"])

scene.describe()

Band names come from the asset keys, and the item's attributes travels with the dataset.

In [ ]:
print("band names:", scene.band_names)
print("timestamp :", scene.timestamp)
print("\nattrs:")
for key, value in scene.attrs.items():
    print(f"  {key}: {value}")

In [ ]:
scene.plot_composite(
    ["red", "green", "blue"],
    pmin=2,
    pmax=98,
    title=f"Sentinel-2 {item.timestamp:%Y-%m-%d}",
)

## Straight into analysis

The loaded dataset is an ordinary `EEORasterDataset`, so every operation applies.

In [ ]:
ndvi = scene.ndvi(red="red", nir="nir")
ndvi.plot_raster_with_histogram(
    cmap="RdYlGn", bins=80, title=f"NDVI - {item.timestamp:%Y-%m-%d}"
)

## Saving the file

The loaded images or hte result (ndvi) in this case can be saved to disk by calling the `.save_raster()` method on any of them (or and `EEORasterDataset`)

In [ ]:
ndvi.save_raster("ndvi_from_stac.tif")

The whole workflow, start to finish:

In [ ]:
(
    eeo.stac_search(
        "sentinel-2-l2a", 
        bbox=bbox, 
        datetime="2026-08-01/..", 
        cloud_cover=10, 
        limit=1
    )[-1]
    .load(["B04", "B08"])
    .ndvi(red="B04", nir="B08")
    .plot_raster(cmap="RdYlGn", title="NDVI from a STAC search")
)

## Other catalogs and collections

`stac_search` defaults to Microsoft Planetary Computer and signs its asset URLs automatically. Point `catalog=` at any STAC API; `sign=False` disables signing for catalogs that do not need it.

```python
results = eeo.stac_search(
    "sentinel-2-l2a",
    bbox=bbox,
    datetime="2026-08-01/..",
    catalog="https://earth-search.aws.element84.com/v1",
    sign=False,
)
```

Landsat works the same way - only the collection id and asset names change (Landsat's NIR is `nir08`, its red is `red`):

```python
landsat = eeo.stac_search(
    "landsat-c2-l2", 
    bbox=bbox, 
    datetime="2026-08-01/..",
    cloud_cover=10, 
    limit=5
)
scene = landsat[-1].load(["red", "nir08"])
ndvi = scene.ndvi(red="red", nir="nir08")
```